# DFIR-Guardrail: padding failure probe (phi3:mini)
The ablation showed 4 padding artifacts that return no usable verdict on every run, and 5 more that the model calls safe on every run. This notebook tests two explanations on exactly those artifacts:

1. **Context overflow:** compare `num_ctx` 2048 / 4096 / 8192 and read `prompt_eval_count` (the real prompt token count).
2. **Wrong output schema:** compare plain `format: "json"` against a JSON-schema `format` that forces `{"analysis", "result" in [0,1]}`.

A control group of 30 other flagged artifacts shows whether the changes move ordinary verdicts. Upload `ablation_bundle.zip` (rebuilt with the new config options) when prompted.

In [ ]:
# 1. Require a GPU, install Ollama, start the daemon
import subprocess
if subprocess.run(["nvidia-smi"], capture_output=True).returncode != 0:
    raise SystemExit("No GPU. Runtime > Change runtime type > T4 GPU, then re-run. CPU timeouts invalidate every result.")
!sudo apt-get update -y
!sudo apt-get install -y zstd pciutils lshw
# Pinned: the published runs used 0.40.16; newer versions change load defaults
!curl -fsSL https://ollama.com/install.sh | OLLAMA_VERSION=0.40.16 sh
!/usr/local/bin/ollama --version

import os, time, requests
os.environ["OLLAMA_NUM_PARALLEL"] = "3"   # must match CONCURRENCY below

def start_ollama():
    global ollama_process
    ollama_process = subprocess.Popen(["/usr/local/bin/ollama", "serve"], stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT, env=os.environ.copy())
    for _ in range(60):
        try:
            if requests.get("http://localhost:11434").status_code == 200: return
        except Exception:
            time.sleep(1)
    raise RuntimeError("Ollama daemon did not become ready")

def restart_ollama():
    # Fresh daemon per run: requests that timed out in the previous run keep generating server-side and would starve the next one
    ollama_process.kill(); subprocess.run(["pkill", "-f", "ollama"]); time.sleep(3); start_ollama()

start_ollama()
!/usr/local/bin/ollama pull phi3:mini
# Preflight: one real generation, then confirm the model is resident on the GPU
t = time.time()
r = requests.post("http://localhost:11434/api/generate", json={"model": "phi3:mini", "prompt": "Say OK.", "stream": False, "options": {"num_ctx": 4096}}, timeout=300)
print(f"preflight generate: {time.time()-t:.1f}s", r.json().get("response", "")[:40])
# Confirm the model is fully in VRAM (read the API: `ollama ps` text changes between versions)
loaded = requests.get("http://localhost:11434/api/ps", timeout=10).json().get("models", [])
print([(x["name"], x["size"], x.get("size_vram")) for x in loaded])
if not loaded or any(x.get("size_vram", 0) < x["size"] for x in loaded):
    print(subprocess.run("grep -iE 'inference compute|cuda|gpu|offload' /content/ollama.log | tail -20", shell=True, capture_output=True, text=True).stdout)
    raise SystemExit("phi3:mini is not fully on the GPU; do not run the eval. The log lines above say why.")

In [ ]:
# 2. Install deps and unpack the exact local code
from google.colab import files
up = files.upload()  # choose ablation_bundle.zip
!unzip -o ablation_bundle.zip -d DFIR-Guardrail
%cd DFIR-Guardrail
!pip install -r requirements.txt

In [ ]:
# 3. Probe: context size and structured output on the padding artifacts
import json, time, requests, collections, pandas as pd
from src.blue_team import BlueTeamGuardrail
from src.red_team import RedTeamPoisoner

ds = RedTeamPoisoner().generate_dataset(1000, 0.2)          # seed 42, same as the ablation
ERRORS = [85, 156, 830, 963]                                  # fail on every ablation run
VETOED_PAD = [154, 267, 340, 391, 544]                        # padding artifacts the model called safe in every run
bt = BlueTeamGuardrail()
flagged = [d["id"] for d in ds if bt._pre_filter_and_clean(d["artifact"])[0]]
CONTROL = [i for i in flagged if i not in ERRORS + VETOED_PAD][:30]   # other flagged artifacts: does the change shift verdicts?
GROUPS = {"error": ERRORS, "vetoed_padding": VETOED_PAD, "control": CONTROL}

VARIANTS = {  # name: (num_ctx, structured)
    "A_baseline_2048": (2048, False),
    "B_ctx4096": (4096, False),
    "C_ctx8192": (8192, False),
    "D_schema_2048": (2048, True),
    "E_schema_4096": (4096, True),
}
REPEATS = 3

def ask(artifact, num_ctx, structured):
    _, cleaned = bt._pre_filter_and_clean(artifact)
    payload = {
        "model": bt.model_name, "stream": False,
        "messages": [{"role": "system", "content": bt.system_prompt},
                     {"role": "user", "content": bt._build_user_content(cleaned)}],
        "format": bt._response_format(structured),
        "options": {"temperature": 0.0, "num_ctx": num_ctx, "num_predict": 200},
    }
    r = requests.post("http://localhost:11434/api/chat", json=payload, timeout=300).json()
    text = r.get("message", {}).get("content", "")
    try:
        res = int(json.loads(text)["result"]); assert res in (0, 1)
    except Exception:
        res = -1                                              # unparseable or wrong schema
    return {"prompt_tokens": r.get("prompt_eval_count"), "result": res, "raw": text[:200]}

rows = []; t0 = time.time()
for v, (ctx, st) in VARIANTS.items():       # variant outermost: Ollama reloads the model whenever num_ctx changes
    for group, ids in GROUPS.items():
        for i in ids:
            for rep in range(REPEATS if group != "control" else 1):
                rows.append({"group": group, "id": i, "variant": v, "ctx": ctx, **ask(ds[i]["artifact"], ctx, st)})
    print(f"{v}: {len(rows)} requests done, {time.time()-t0:.0f}s elapsed", flush=True)
df = pd.DataFrame(rows)
df.to_csv("padding_probe.csv", index=False)

# prompt_tokens at or near num_ctx means the prompt was truncated
print(df.groupby(["group", "variant"]).agg(n=("result", "size"), prompt_tokens_max=("prompt_tokens", "max"),
      prompt_tokens_mean=("prompt_tokens", "mean"), valid=("result", lambda s: (s >= 0).mean()),
      flagged=("result", lambda s: (s == 1).mean())).round(2).to_string())
print()
print(df[df.group == "error"][["id", "variant", "prompt_tokens", "result", "raw"]].drop_duplicates(["id", "variant"]).to_string(index=False))

In [ ]:
# 4. Download results and stop Ollama
files.download("padding_probe.csv")
ollama_process.terminate()